# Lab 04 · Tool calling: an agent with three tools

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` and save its key in Colab Secrets. DeepSeek uses `DEEPSEEK_API_KEY`; Gemini and FreeInference use their respective secrets. `client` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import Client, PROVIDERS, secret
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
settings = PROVIDERS[PROVIDER]
client = Client(model=settings["model"], family=settings["family"],
                base_url=settings["base_url"], api_key=secret(settings["secret_name"]),
                max_calls=100, max_tokens=256, extra_body=settings.get("extra_body"))
ask = client.ask

## Warm-up · Give the model a small tool

### Warm-up step 1 · Ask without tools

Ask for the current time and weather in Rome. The prompt supplies no clock or weather reading. Compare the answer with a source you trust, even if it sounds plausible.

In [ ]:
prompt = "What is the current time and weather in Rome, Italy?"
answer = ask(prompt)
print("Question:", prompt)
print("Model answer without tools:", answer)

### Warm-up step 2 · Give the model a weather tool

Tell the model that `get_weather` is available and ask which action it wants. If the reply names that function, Python calls the public Open-Meteo service and prints the returned data. The next prompt includes that data so the model can report the temperature, observation time, and weather code.

The model may return `no_tool` or something Python cannot use. Read `Model request` first, then rerun the cell if you want to try again. The weather service also needs network access. If the lookup fails, continue with the main exercise, which uses sample records in Python.

In [ ]:
from urllib.request import urlopen
import json

def get_weather():
    url = ("https://api.open-meteo.com/v1/forecast?latitude=41.9028"
           "&longitude=12.4964&current=temperature_2m,weather_code"
           "&timezone=Europe%2FRome")
    with urlopen(url, timeout=15) as response:
        return json.load(response)["current"]

choice = ask("Available tool: get_weather (current weather in Rome). "
             "Question: What is the weather in Rome? Reply only get_weather or no_tool.")
print("Model request:", choice)
if choice.strip().lower().strip(' ."') == "get_weather":
    weather = get_weather()
    print("Live tool result:", weather)
    print("Model answer:", ask(f"Current weather in Rome: {weather}. State the temperature in °C and observation time. Report the weather code; do not guess its meaning."))
else:
    print("The model did not request the tool. Run this cell again or inspect its reply.")

**Next:** Now give an agent three small tools for working with microservice information.

## Give an agent three small tools
**Problem:** A language model can describe an Orders service, but it needs the real record to answer questions about your Orders service.

You give it three functions that look up a record, check its fields, and format it for a web page. You first call each function directly in Python. Then you let the model request the functions and inspect what Python runs. The main exercise uses two sample records stored in the notebook.

**At a glance**

![Lab 04 at a glance: Question, then Agent chooses (system lists tools), then Extract (service record), then Validate (required fields), then Format HTML (escaped text), then Answer (from tool results).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/04.svg)

## Think about the example

A microservice owns its data and exposes information through a defined interface. The local service records in this example are illustrative.

**Concepts:** Database per service; Service-owned data; Published interface.

Use the scenario above to consider these questions before running the code:

1. Why should a service expose its data through an interface rather than let other services access its database directly?
2. What does checking that a service record has required fields tell you, and what does it not prove?

## The idea in context

A tool is an ordinary function the agent can request. The application runs it and returns its result.

A microservice owns its data and exposes useful information through an interface.

| Key term | Meaning |
| --- | --- |
| **Tool** | A function the model can ask the application to run. |
| **System prompt** | Instructions that describe the agent and its available tools. |
| **Validation** | Checking whether data satisfies defined rules. |
| **Tool result** | The information returned to the model after a tool runs. |

## Pattern · Tool-using agent
A **tool** is a function the application makes available for the model to request. The model sends a request such as `extract_service` with the name `orders`. Python decides whether the request is allowed, runs the function, and includes the returned record in the next prompt. The combination of a model, tools, and a loop that passes along results is the agent in this lab.

## Main exercise at a glance

1. Create a tool to extract data
2. Create a tool to validate data
3. Create a tool to format HTML
4. Tell the agent about its tools
5. Let the agent choose and use tools

## Step 1 · Create a tool to extract data
Run the function directly before giving it to the model. `extract_service("orders")` reads the Orders record from the local `services` dictionary. Expect to see its name, owner and responsibility. Try `extract_service("shipping")` to see the empty dictionary returned for a missing service.

In [ ]:
services = {
    "orders": {"name": "Orders", "owner": "Sales team", "responsibility": "Manage orders"},
    "inventory": {"name": "Inventory", "owner": "Stock team", "responsibility": "Track stock"},
}

def extract_service(name: str) -> dict:
    """Find an Orders or Inventory record by name in the local catalog."""
    return services.get(name.lower().strip(), {}).copy()

print("Extracted:", extract_service("orders"))

## Step 2 · Create a tool to validate data
**Validation** means checking a record against explicit rules. This function requires a name, owner, and responsibility, each with nonempty text. Run the two examples. A complete record prints `Valid` and a record with no owner prints `Invalid: missing owner`. These checks verify the fields, and you still need a trusted source to confirm the details.

In [ ]:
def validate_service(data: dict) -> str:
    """Check that a service record has a name, owner, and responsibility."""
    for field in ("name", "owner", "responsibility"):
        if not isinstance(data.get(field), str) or not data[field].strip():
            return "Invalid: missing " + field
    return "Valid"

print("Complete record:", validate_service(extract_service("orders")))
print("Missing owner:", validate_service({"name": "Orders"}))

## Step 3 · Create a tool to format HTML
HTML is text with tags that a web browser uses to display content. The tool uses `<h2>` for the service name and `<p>` for paragraphs. It checks the record again before building the HTML. `escape` keeps characters such as `<` in service data from being interpreted as tags.

In [ ]:
from html import escape

def format_html(data: dict) -> str:
    """Format a valid service record as HTML; return an error if invalid."""
    if validate_service(data) != "Valid":
        return "Cannot format an invalid record."
    name = escape(data["name"])
    owner = escape(data["owner"])
    responsibility = escape(data["responsibility"])
    return f"<h2>{name}</h2><p>Owner: {owner}</p><p>Responsibility: {responsibility}</p>"

print(format_html(extract_service("orders")))

## Step 4 · Tell the agent about its tools
The **system prompt** describes the available tools and asks the model to send one JSON object per turn. For example, `{"tool": "extract_service", "name": "orders"}` names the function and its argument. Python reads `tool` to decide which branch to run.

`html_target` identifies the service that Python allows this run to look up and format. Changing it also changes the question. For a conceptual question, set `html_target = None` and edit the alternate question. In that mode the model can give a plain explanation without any tools.

In [ ]:
import json

system_prompt = """You are a helpful microservices assistant.
Choose one action at a time. Reply with a JSON object only:
{"tool": "extract_service", "name": "the requested service"}
{"tool": "validate_service"}
{"tool": "format_html"}
{"tool": "finish", "answer": "explanation for a conceptual question"}
For HTML requests, extract the requested record, validate it, then format it.
Use finish after formatting or observing an error for that record.
Python supplies the HTML result. For conceptual questions, finish with an answer.
"""
html_target = "Orders"
question = f"Show the {html_target} service as HTML." if html_target else "What does a microservice own?"
print("User question:", question)

def read_action(reply):
    try:
        action = json.loads(reply)
    except json.JSONDecodeError:
        return {}
    return action if isinstance(action, dict) else {}

## Step 5 · Let the agent choose and use tools
Each turn sends the question and `history`, the requests and results recorded so far. Python runs an allowed function and adds its result to the next prompt. A wrong service lookup, malformed reply or out-of-order action gives the model an error it can use on the next turn. Every rerun starts with an empty history.

For HTML, Python allows only the requested service. A new lookup clears validation and any earlier result. The model can finish after Python has formatted the validated record or observed why that record is unavailable. The displayed answer comes directly from Python in either case. The loop has a six-turn budget.

In [ ]:
data, validated, output, history = {}, False, None, ""
for turn in range(6):
    reply = client.ask(f"Question: {question}\nPrevious results:\n{history}", system=system_prompt)
    print("\nAgent:", reply)
    action = read_action(reply)
    tool = action.get("tool")
    if tool == "extract_service" and isinstance(action.get("name"), str):
        name = action["name"].strip().lower()
        data, validated, output = {}, False, None
        if html_target and name != html_target.strip().lower():
            result = f"Wrong service. Look up {html_target}."
        else:
            data = extract_service(name)
            result = data or "Service not found."
            output = None if data else result
    elif tool == "validate_service" and data:
        result = validate_service(data)
        validated = result == "Valid"
        output = None if validated else result
    elif tool == "format_html" and validated:
        result = output = format_html(data)
    elif tool == "finish" and isinstance(output if html_target else action.get("answer"), str):
        print("Final answer:", output if html_target else action["answer"])
        break
    else:
        result = "Use one JSON object. For HTML, extract and validate, then format before finish."
    print("Tool result:", result)
    history += f"{reply}\nResult: {result}\n"
else:
    print("Stopped after six turns. Inspect the trace and try again.")

## What you should see
Look for the Orders record, `Valid`, and the formatter's HTML. On `finish`, Python displays that same HTML so the final output stays tied to the record. The notebook shows the HTML as text so you can read the tags. The **trace** is the printed sequence of requests and results.

If a lookup or validation fails, the final output can show the tool's error for the requested record. If the loop uses all six turns, read the trace and rerun Step 5 to start over.

## Try it
1. Set `html_target = "Inventory"` in Step 4. Run Steps 4 and 5 and check the owner in the final output.
2. Set `html_target = "Shipping"` and rerun Steps 4 and 5. What does the catalog lookup return?
3. Remove the Orders owner in Step 1, set the target back to `"Orders"`, and rerun Steps 1–5. What does validation report?
4. Set `html_target = None` in Step 4 and run Steps 4 and 5. Check whether the trace shows any tools when the model answers “What does a microservice own?”

## Takeaway
The model proposes actions. Python controls which functions run, which record is allowed, and what counts as a completed HTML request.

## Optional · The same tools with LangGraph

In Step 5, you wrote the loop yourself. Here, LangChain's `create_agent` builds a **LangGraph** workflow that manages tool requests and results. The example uses the same catalog and validation functions. A wrapper checks that the record supplied for formatting matches the requested catalog entry.

Run Setup and Steps 1–3, then run the install and example cells. This version needs a model that supports **native tool calling**, where the provider returns a structured tool request. Use the manual example if your provider rejects native calls or the optional packages conflict. This section uses a separate model client with its own workflow limit.

In [ ]:
%pip install -q "langchain==1.5.2" "langchain-openai==1.8.5"

example_target = "Orders"

def format_catalog_html(data: dict) -> str:
    """Format the requested service only if the data matches the catalog."""
    catalog_record = extract_service(example_target)
    if data != catalog_record:
        return f"Extract the {example_target} record from the catalog first."
    return format_html(catalog_record)

### Create and run the agent

The framework runs requested functions and returns their results to the model. Print the messages to see whether the model used the tools in order. The wrapper checks the supplied data against the catalog, then the original formatter checks its fields. The final output shows checked HTML only when the formatter returned it.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import ToolMessage
model = ChatOpenAI(
    model=settings["model"], base_url=settings["base_url"],
    api_key=secret(settings["secret_name"]), extra_body=settings.get("extra_body"), max_tokens=256,
)
agent = create_agent(
    model=model, tools=[extract_service, validate_service, format_catalog_html],
    system_prompt="Extract the requested service, validate its record, and call format_catalog_html.",
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": f"Show the {example_target} service as HTML."}]},
    config={"recursion_limit": 12},
)
checked_html = None
for message in result["messages"]:
    if getattr(message, "tool_calls", None):
        print("Agent requested:", message.tool_calls)
    if isinstance(message, ToolMessage):
        print("Tool result:", message.content)
        if message.name == "format_catalog_html" and message.content.startswith("<h2>"):
            checked_html = message.content
print("Final output:", checked_html or "No checked HTML produced.")

**Compare.** The manual example uses `if` statements to enforce the sequence. The framework example manages the loop and checks the formatter's input against the catalog. Inspect the trace to see which tools the model requested. Change `example_target` to `"Inventory"` or `"Shipping"` in the optional setup cell and rerun both optional cells.